# Black-Box Optimisation Dashboard — Data Prototyping

This notebook is the **scratchpad** where we explore `weekly_optimisation_summary.xlsx`
and build each chart *before* it gets moved into the Streamlit app (`app.py` /
`utils.py`, in the `bbo_dashboard/` folder).

Working this way — notebook first, app second — is a good habit: a notebook lets you
run one cell at a time and immediately see a table or a chart, which is much faster
for exploring than re-running a whole Streamlit app on every change. Once a chunk of
code does what you want, you "graduate" it into a `.py` file.

**What's in the data?**
- `weeksummary` sheet — one row per (Week, Function) submission: the query point
  submitted, the model's predicted value, the actual output returned, and notes.
- `result` sheet — one row per Function, with its actual output for every week,
  plus initial best / final best / leaderboard position.

8 functions × 13 weeks.


In [1]:
# If a package below is missing, install it from a terminal with:
#   pip install pandas openpyxl matplotlib plotly
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

DATA_PATH = "data/weekly_optimisation_summary.xlsx"

# Always good practice: see what sheets exist before assuming anything
xl = pd.ExcelFile(DATA_PATH)
xl.sheet_names

['weeksummary', 'result']

## 1. Load the `weeksummary` sheet

The real column headers sit on the **second** row of the sheet (the first row is a
merged title cell), so we pass `skiprows=1` to skip past it.

In [2]:
weeksummary = pd.read_excel(DATA_PATH, sheet_name="weeksummary", skiprows=1)
weeksummary["Week"] = weeksummary["Week"].astype(int)
weeksummary["Function"] = weeksummary["Function"].astype(int)

print(weeksummary.shape)
weeksummary.head()

(104, 11)


,Week,Function,Function Description,Submitted Query (x),Model's Predicted Value / Acquisition Score,Returned Actual Output,Initial Best Output,Actual Output Above Initial Best,Candidate Generator / Surrogate Model,Notes,Source Notebook
0,1,1,"Radiation field (2D, 10 initial samples, hairl...",NaN,NaN,NaN,7.700000e-16,NaN,NaN,"n/a, revisited via tutorial",w1-functions-all.ipynb and w2-functions-all.ipynb
1,1,2,"Noisy log-likelihood field (2D, 10 initial sam...",NaN,NaN,NaN,6.112050e-01,NaN,NaN,"n/a, revisited via tutorial",w1-functions-all.ipynb and w2-functions-all.ipynb
2,1,3,"Drug discovery project (3D, 15 initial samples...",NaN,NaN,NaN,-3.483500e-02,NaN,NaN,"n/a, revisited via tutorial",w1-functions-all.ipynb and w2-functions-all.ipynb
3,1,4,"Warehouse business (4D, 30 initial samples, hi...",NaN,NaN,NaN,-4.025542e+00,NaN,NaN,"n/a, revisited via tutorial",w1-functions-all.ipynb and w2-functions-all.ipynb
4,1,5,"Chemical process in a factory (4D, 20 initial ...",NaN,NaN,NaN,1.088860e+03,NaN,NaN,"n/a, revisited via tutorial",w1-functions-all.ipynb and w2-functions-all.ipynb


In [3]:
# Quick sanity checks — always worth doing on a new dataset
print("Weeks:", sorted(weeksummary['Week'].unique()))
print("Functions:", sorted(weeksummary['Function'].unique()))
print("\nMissing values per column:")
weeksummary.isna().sum()

Weeks: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
Functions: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

Missing values per column:


Week                                            0
Function                                        0
Function Description                            0
Submitted Query (x)                             8
Model's Predicted Value / Acquisition Score    16
Returned Actual Output                          8
Initial Best Output                             0
Actual Output Above Initial Best                8
Candidate Generator / Surrogate Model          16
Notes                                          64
Source Notebook                                 0
dtype: int64

Week 1 has no `Submitted Query`, `Predicted Value`, or `Actual Output` —
that's expected: week 1 was the *initial sampling* phase before any optimisation
submissions were made, so those cells are blank on purpose.

## 2. Load the `result` sheet

This sheet is "wide": one column per week (`Week 1`, `Week 2`, ... `Week 13`).
Wide format is easy to read as a human, but hard to plot — a plotting library wants
one row per (Function, Week) pair instead. We'll reshape it with `pandas.melt`.

In [4]:
results = pd.read_excel(DATA_PATH, sheet_name="result", skiprows=1)
results["Function"] = results["Function"].astype(int)
results

,Function,Function Description,Archetype,Dimension,Initial Sample,Final Sample,Week 1,Week 2,Week 3,Week 4,...,Week 7,Week 8,Week 9,Week 10,Week 11,Week 12,Week 13,Initial Best,BBO Best,Leaderboard
0,1,Radiation field,Hairline_spike,2,10,22,NaN,1.331915e-22,2.040000e-50,-3.890649e-08,...,-0.000007,2.009599e-19,6.880969e-15,3.172948e-11,3.239181e-13,5.000737e-13,9.921997e-09,7.700000e-16,9.920000e-09,15
1,2,Noisy log-likelihood field,Rising_edge,2,10,22,NaN,-4.065150e-02,4.687417e-01,4.875634e-01,...,0.563547,6.831875e-01,6.360729e-01,5.109425e-01,6.135463e-01,5.805297e-01,6.735663e-01,6.112050e-01,6.831880e-01,11
2,3,Drug discovery project,Corner_lock,3,15,27,NaN,-5.061472e-02,-1.950802e-02,-1.194251e-01,...,-0.069599,-2.344783e-02,-2.926924e-02,-1.851607e-02,-1.811187e-02,-3.816399e-02,-3.348463e-02,-3.483500e-02,-1.507200e-02,16
3,4,Warehouse business,Hidden_bowl,4,30,42,NaN,-7.132125e+00,2.037936e-01,-9.915951e-01,...,0.657359,6.640783e-01,4.721300e-01,2.736215e-01,-1.334416e+00,1.918038e-01,5.113392e-01,-4.025542e+00,6.640780e-01,4
4,5,Chemical process in a factory,Edge_surge,4,20,32,NaN,1.829810e+03,7.106023e+02,2.699104e+03,...,4461.243621,4.826283e+03,5.022768e+03,1.465336e+04,6.469995e+03,8.232981e+03,8.662483e+03,1.088860e+03,8.662483e+03,1
5,6,Cake recipe,Noise_ceiling,5,20,32,NaN,-1.241946e+00,-3.529449e-01,-8.372998e-01,...,-0.859304,-6.507745e-01,-3.383813e-01,-1.088831e+00,-1.320646e+00,-7.456966e-01,-3.940346e-01,-7.142650e-01,-3.383810e-01,17
6,7,Tuning six hyperparameters,Ghost_spike,6,30,42,NaN,5.927941e-01,2.594798e+00,1.088041e+00,...,2.063588,1.642929e+00,2.749208e+00,1.762364e+00,2.627029e+00,2.834909e+00,3.013542e+00,1.364968e+00,3.013542e+00,5
7,8,Unspecified field,Deep_lock,8,40,52,NaN,8.264884e+00,7.350985e+00,9.760084e+00,...,9.638209,9.752723e+00,9.730201e+00,9.908295e+00,9.729285e+00,9.764940e+00,9.991664e+00,9.598482e+00,9.991664e+00,5


In [5]:
week_cols = [f"Week {i}" for i in range(1, 14)]

results_long = results.melt(
    id_vars=["Function", "Function Description", "Archetype", "Dimension",
             "Initial Sample", "Final Sample", "Initial Best", "BBO Best", "Leaderboard"],
    value_vars=week_cols,
    var_name="Week",
    value_name="Actual Output",
)

# "Week 7" -> 7
results_long["Week"] = results_long["Week"].str.replace("Week ", "", regex=False).astype(int)

# Week 1 is 'n/a' for every function (no submission yet) -> drop those rows
results_long = results_long[pd.to_numeric(results_long["Actual Output"], errors="coerce").notna()]
results_long["Actual Output"] = results_long["Actual Output"].astype(float)
results_long = results_long.sort_values(["Function", "Week"]).reset_index(drop=True)

results_long.head(10)

,Function,Function Description,Archetype,Dimension,Initial Sample,Final Sample,Initial Best,BBO Best,Leaderboard,Week,Actual Output
0,1,Radiation field,Hairline_spike,2,10,22,7.700000e-16,9.920000e-09,15,2,1.331915e-22
1,1,Radiation field,Hairline_spike,2,10,22,7.700000e-16,9.920000e-09,15,3,2.040000e-50
2,1,Radiation field,Hairline_spike,2,10,22,7.700000e-16,9.920000e-09,15,4,-3.890649e-08
3,1,Radiation field,Hairline_spike,2,10,22,7.700000e-16,9.920000e-09,15,5,5.184470e-188
4,1,Radiation field,Hairline_spike,2,10,22,7.700000e-16,9.920000e-09,15,6,8.074692e-24
5,1,Radiation field,Hairline_spike,2,10,22,7.700000e-16,9.920000e-09,15,7,-7.447376e-06
6,1,Radiation field,Hairline_spike,2,10,22,7.700000e-16,9.920000e-09,15,8,2.009599e-19
7,1,Radiation field,Hairline_spike,2,10,22,7.700000e-16,9.920000e-09,15,9,6.880969e-15
8,1,Radiation field,Hairline_spike,2,10,22,7.700000e-16,9.920000e-09,15,10,3.172948e-11
9,1,Radiation field,Hairline_spike,2,10,22,7.700000e-16,9.920000e-09,15,11,3.239181e-13


## 3. Progress of a single function, over time

Let's plot **Function 5** (Chemical process in a factory) as an example — pick any
function number and re-run the cell.

In [6]:
FUNCTION = 5

fn_row = results[results["Function"] == FUNCTION].iloc[0]
fn_progress = results_long[results_long["Function"] == FUNCTION]

fig = go.Figure()
fig.add_trace(go.Scatter(x=fn_progress["Week"], y=fn_progress["Actual Output"],
                          mode="lines+markers", name="Actual output"))
fig.add_hline(y=fn_row["Initial Best"], line_dash="dash", line_color="gray",
              annotation_text="Initial best")
fig.update_layout(title=f"Function {FUNCTION} — {fn_row['Function Description']}",
                   xaxis_title="Week", yaxis_title="Actual output", xaxis=dict(dtick=1))
fig.show()

## 4. Comparing all 8 functions on one chart

Problem: the 8 functions live on wildly different scales (Function 1 hovers near
`1e-16`, Function 5 climbs into the thousands). Plotting raw values together would
make 7 of the 8 lines look completely flat next to Function 5.

**Fix:** min-max scale each function's own weekly values to a 0–1 range. This throws
away the absolute numbers but lets us compare the *shape* of each function's
progress — fast improvers vs. slow, steady vs. jumpy — fairly, on one chart.

In [7]:
def normalise(group):
    lo, hi = group.min(), group.max()
    if hi == lo:
        return pd.Series(0.5, index=group.index)
    return (group - lo) / (hi - lo)

results_long["Normalised Output"] = results_long.groupby("Function")["Actual Output"].transform(normalise)

fig = px.line(
    results_long.assign(**{"Function": "F" + results_long["Function"].astype(str)}),
    x="Week", y="Normalised Output", color="Function", markers=True,
)
fig.update_layout(title="Normalised progress, all functions", xaxis=dict(dtick=1))
fig.show()

## 5. Leaderboard

A simple horizontal bar chart, sorted so the "best" position (as recorded in the
sheet — check your challenge rules for whether lower or higher means first place)
is easy to scan.

In [8]:
lb = results[["Function", "Function Description", "Leaderboard"]].sort_values("Leaderboard")
lb_label = "F" + lb["Function"].astype(str) + " — " + lb["Function Description"]

fig = px.bar(lb.assign(Label=lb_label).sort_values("Leaderboard", ascending=False),
             x="Leaderboard", y="Label", orientation="h")
fig.update_layout(title="Leaderboard position by function", yaxis_title="")
fig.show()

## 6. Turning free-text notes into a 'model family' column

The `Candidate Generator / Surrogate Model` column is free text, e.g.
`"GP (Matern ARD) - UCB"` or `"MC-dropout NN (weighted calibration) - UCB"`.
That's great for a human reading the log, but useless for a bar chart — we need a
short, consistent label instead. Below is a small rule-based classifier: it checks
each note against a list of patterns (in priority order) and returns a family name
like `"Gaussian Process"` or `"Random Forest"`.

A neat trick used here: matching `"rf"` or `"gp"` as a *whole word* with a regex
(`\bgp\b`) instead of a plain substring check — otherwise `"gp"` would wrongly
match inside unrelated words, and `"rf"` would match inside `"overfitting"`.

In [9]:
import re

def _word(s, w):
    return re.search(rf"\b{re.escape(w)}\b", s) is not None

FAMILY_RULES = [
    (lambda s: "bake-off" in s, "Model Bake-off"),
    (lambda s: "trust-region" in s, "Trust Region + EI"),
    (lambda s: "loocv" in s and "best model" in s, "Model Selection (LOOCV)"),
    (lambda s: "local surrogate" in s or "local bootstrap-gp" in s, "Local Surrogate"),
    (lambda s: "mc-dropout" in s or "dropout nn" in s or _word(s, "mlp"), "NN (MC-Dropout / MLP)"),
    (lambda s: _word(s, "gp") and _word(s, "rf"), "GP + RF Hybrid"),
    (lambda s: _word(s, "gp") and _word(s, "svr"), "GP + SVR Blend"),
    (lambda s: _word(s, "gp") and _word(s, "svm"), "GP + SVM"),
    (lambda s: _word(s, "gp") and "polynomial" in s, "GP + Polynomial"),
    (lambda s: _word(s, "gp") or "gaussian" in s, "Gaussian Process"),
    (lambda s: "extratrees" in s, "ExtraTrees"),
    (lambda s: _word(s, "rf") or "random forest" in s, "Random Forest"),
    (lambda s: "tree ensemble" in s or "bootstrap tree" in s, "Tree Ensemble"),
    (lambda s: _word(s, "svr"), "SVR"),
]

def classify_generator(text):
    if not isinstance(text, str) or not text.strip():
        return "Not recorded"
    s = text.lower()
    for test, label in FAMILY_RULES:
        if test(s):
            return label
    return "Other"

weeksummary["Model Family"] = weeksummary["Candidate Generator / Surrogate Model"].apply(classify_generator)
weeksummary["Model Family"].value_counts()

Model Family
Gaussian Process           34
Not recorded               16
GP + SVR Blend             13
NN (MC-Dropout / MLP)      10
GP + RF Hybrid              9
Random Forest               9
GP + Polynomial             2
Tree Ensemble               2
GP + SVM                    2
Local Surrogate             2
ExtraTrees                  1
SVR                         1
Trust Region + EI           1
Model Bake-off              1
Model Selection (LOOCV)     1
Name: count, dtype: int64

In [10]:
fam_counts = weeksummary["Model Family"].value_counts().reset_index()
fam_counts.columns = ["Model Family", "Count"]

fig = px.bar(fam_counts.sort_values("Count"), x="Count", y="Model Family", orientation="h")
fig.update_layout(title="Candidate generator families used across all submissions")
fig.show()

## 7. From notebook to Streamlit app

Every idea above — loading the two sheets, reshaping to long format, the
normalisation trick, the family classifier — has been copied into
**`bbo_dashboard/utils.py`**, wrapped in `@st.cache_data` so Streamlit only re-reads
the Excel file once instead of on every click.

**`bbo_dashboard/app.py`** then just calls those functions and lays out six pages
(Overview, Function Progress, Compare Functions, Leaderboard, Candidate Generators,
Submission Log) using `st.selectbox`, `st.multiselect`, `st.plotly_chart`, etc.

To run the live dashboard from a terminal:
```bash
cd bbo_dashboard
pip install -r requirements.txt
streamlit run app.py
```
See `README.md` in that folder for full setup instructions.
